In [15]:
import pandas as pd
import numpy as np

import joblib
import json

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

from sklearn.preprocessing import LabelEncoder

In [16]:
df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 128)


In [17]:
df["international_market_access"] = np.where(

    df["international_presence"] > 0,

    1,

    0

)

In [18]:
def market_label(score):

    if score < 35:
        return "Low"

    elif score < 60:
        return "Medium"

    else:
        return "High"


df["market_opportunity_label"] = (
    df["market_opportunity_score"]
    .apply(market_label)
)

print(
    df["market_opportunity_label"]
    .value_counts()
)

market_opportunity_label
Low       39843
Medium    10035
High        122
Name: count, dtype: int64


In [19]:
features = [

    "market_size",

    "market_growth_rate",

    "tam",

    "sam",

    "som",

    "customer_segment_count",

    "international_market_access",

    "industry_growth_rate"

]

X = df[features]

y = df["market_opportunity_label"]

In [20]:
le = LabelEncoder()

y = le.fit_transform(y)

print(
    dict(
        zip(
            le.classes_,
            range(
                len(le.classes_)
            )
        )
    )
)

{'High': 0, 'Low': 1, 'Medium': 2}


In [21]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [22]:
model = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    random_state=42,

    n_jobs=-1

)

model.fit(
    X_train,
    y_train
)

RandomForestClassifier(max_depth=12, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [23]:
preds = model.predict(X_test)

acc = accuracy_score(
    y_test,
    preds
)

print(
    "Accuracy:",
    round(acc*100,2),
    "%"
)

print(
    classification_report(
        y_test,
        preds
    )
)

Accuracy: 99.93 %
              precision    recall  f1-score   support

           0       0.95      0.88      0.91        24
           1       1.00      1.00      1.00      7969
           2       1.00      1.00      1.00      2007

    accuracy                           1.00     10000
   macro avg       0.98      0.96      0.97     10000
weighted avg       1.00      1.00      1.00     10000



In [24]:
importance = pd.DataFrame({

    "Feature": features,

    "Importance":
        model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

print(
    importance
)

                       Feature  Importance
1           market_growth_rate    0.968154
2                          tam    0.006908
0                  market_size    0.006297
4                          som    0.006256
3                          sam    0.005676
7         industry_growth_rate    0.004339
5       customer_segment_count    0.001699
6  international_market_access    0.000669


In [26]:
joblib.dump(

    model,

    "../models/market_opportunity_model/market_opportunity_model.pkl"

)

joblib.dump(

    le,

    "../models/market_opportunity_model/label_encoder.pkl"

)

print("Market Opportunity Model Saved ✅")

Market Opportunity Model Saved ✅


In [27]:
metadata = {

    "model_name":
        "Market Opportunity Model",

    "algorithm":
        "Random Forest",

    "features":
        features,

    "classes":
        list(
            le.classes_
        )

}

with open(

    "../models/market_opportunity_model/metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

In [28]:
df.to_csv(
    "../datasets/cleaned/startup_info.csv",
    index=False
)

print("Dataset Saved ✅")

Dataset Saved ✅


In [29]:
print(df["market_opportunity_label"].value_counts())

print("Accuracy:", acc)

print(importance)

market_opportunity_label
Low       39843
Medium    10035
High        122
Name: count, dtype: int64
Accuracy: 0.9993
                       Feature  Importance
1           market_growth_rate    0.968154
2                          tam    0.006908
0                  market_size    0.006297
4                          som    0.006256
3                          sam    0.005676
7         industry_growth_rate    0.004339
5       customer_segment_count    0.001699
6  international_market_access    0.000669


In [ ]:
for col in features:
    print("\n", col)
    print(df[col].dtype)


 market_size
int64

 market_growth_rate
float64

 tam
int64

 sam
int64

 som
int64

 customer_segment_count
int64

 international_market_access
object

 industry_growth_rate
float64


In [ ]:
print(df["international_market_access"].value_counts())
print(df["international_market_access"].unique())

international_market_access
Synthetic    300
826           74
42            72
341           70
493           70
            ... 
400           31
426           31
841           29
656           29
165           27
Name: count, Length: 1001, dtype: int64
['Synthetic' '321' '133' ... '753' '559' '156']


In [ ]:
df["international_market_access"] = np.where(

    df["international_presence"] > 0,

    1,

    0

)

In [ ]:
print(
    df["international_market_access"]
    .value_counts()
)

international_market_access
1    49852
0      148
Name: count, dtype: int64
